# WW-DHSVM — building a DHSVM model of the Connecticut River Basin

**A watershed in, a running hydrologic model out.**

This notebook builds a complete, runnable
[DHSVM](https://dhsvm.pnnl.gov/) simulation of the **entire Connecticut
River Basin** (USGS HUC4 `0108`, 29,186 km² — New England's largest river
system, from the Canadian border to Long Island Sound) starting from
nothing but that four-digit code. Every input DHSVM needs — terrain, channel network,
soils, vegetation, meteorology, initial state and the configuration file
itself — is downloaded from public sources, harmonised onto one model
grid, checked, and written in DHSVM's native formats.

---

## What this tool is, and where it comes from

WW-DHSVM is a **fork of [Watershed
Workflow](https://github.com/environmental-modeling-workflows/watershed-workflow)
v2.1.0** (BSD licensed), the toolset the ATS community uses to
parameterise integrated hydrologic models. Watershed Workflow's central
insight is worth stating plainly, because it is what this fork inherits:

> A hyper-resolution model setup should be **reproducible from a
> watershed identifier**. Everything else — which national dataset, which
> year, which projection, which cache — should be a declarative choice
> made by a *source manager*, not a sequence of manual GIS steps.

What was kept:

* the **source-manager architecture** — abstract base classes that
  normalise CRS handling, buffer and snap bounding boxes, and maintain an
  on-disk cache with *superset detection*, so re-running this notebook
  re-downloads nothing;
* the data-source implementations: WBD, NHD, 3DEP, NLCD, SSURGO,
  SoilGrids, POLARIS, AORC, Daymet;
* CRS, warping and colour infrastructure.

What was **removed** — everything specific to ATS's unstructured
discretisation: 2D/3D mesh generation, Delaunay triangulation,
stream-aligned river meshing, mesh extrusion, ExodusII and VTK writers,
labeled-set regions, and the ATS XML input-spec writers. DHSVM runs on a
single regular grid, so none of it has an analogue.

What was **added** — the DHSVM half of the problem:
`grid`, `binary`, `terrain`, `streams`, `soils`, `vegetation`, `states`,
`meteorology`, `config_writer`, `diagnostics`, `output`, `figures`, and a
new **NOAA HRRR** source manager built on
[Herbie](https://github.com/blaylockbk/Herbie).

## DHSVM and ATS — the same physics, two very different discretisations

Both models solve coupled surface–subsurface hydrology at hyper-resolution,
and both need the same *kinds* of data. They differ in almost everything
else, and those differences are exactly what this fork had to rewrite.

| | **DHSVM** | **ATS** |
|---|---|---|
| **Discretisation** | one regular, north-up, square grid | unstructured, stream-aligned, mixed-polyhedral mesh |
| **Subsurface flow** | quasi-3D: 1D vertical unsaturated + 2D saturated lateral, transmissivity decaying exponentially with depth | fully 3D variably-saturated Richards equation |
| **Surface flow** | explicit cell-to-cell routing (D4 or D8) + a 1D channel network | diffusion-wave overland flow coupled to the subsurface |
| **Canopy** | explicit **two-layer** overstory/understory energy balance | single-layer canopy with land-cover parameters |
| **Snow** | two-layer mass and energy balance, with canopy interception, unloading and (optionally) sliding | multi-layer snowpack |
| **Soil hydraulics** | **Brooks–Corey** (λ, air-entry ψ_b) | **van Genuchten** (α, n) |
| **Time stepping** | fixed timestep, typically 1–3 h | adaptive, sub-second to daily |
| **Input format** | headerless binary maps + ASCII tables + one INI-style config | ExodusII mesh + HDF5 forcing + XML |
| **Parallelism** | Global Arrays + MPI (the `parallel` branch) | MPI throughout |
| **Written in** | C | C++ (on Amanzi/Arcos) |

**Where they overlap** — and therefore where Watershed Workflow's code
transferred directly — is data acquisition: both need a watershed
boundary, a DEM, land cover, soil properties, and meteorological forcing,
all clipped to a domain and reprojected to a common CRS.

**Where they diverge** is everything downstream. ATS asks *"what mesh?"*;
DHSVM asks *"what grid, and what are the 19 soil and 35 vegetation
parameters for each class on it?"*

## What DHSVM actually reads

Nine kinds of file, all of which this notebook writes:

**1. The configuration file** (`INPUT.ConnecticutRiverBasin`) — INI-style
sections `[OPTIONS] [AREA] [TIME] [CONSTANTS] [TERRAIN] [ROUTING]
[METEOROLOGY] [SOILS] [VEGETATION] [OUTPUT]`. Its parser is forgiving
about whitespace and ordering, and completely unforgiving about a key
being missing when an option requires it.

**2–6. Five binary maps** — headerless streams of values in row-major
order from the **north-west** corner. There is no metadata in the file at
all; shape comes from `[AREA]` and dtype from a compiled-in table in
DHSVM's `VarID.c`:

| map | quantity | dtype |
|---|---|---|
| `dem.bin` | elevation (m) | `float32` |
| `mask.bin` | basin mask | `uint8` |
| `soil.bin` | soil class `1..N` | `uint8` |
| `soild.bin` | soil column depth (m) | `float32` |
| `veg.bin` | vegetation class `1..N` | `uint8` |

**7–9. Three stream files** that must agree exactly:
`stream.class.dat` (channel classes), `stream.network.dat` (segment
topology), `stream.map.dat` (which segments cross which grid cells,
with **zero-based** column and row indices).

Plus **meteorological station files**, **initial-state files**, and —
whenever any vegetation type is partly impervious — an
**impervious-surface routing file**.

In [ ]:
# --- setup ---------------------------------------------------------------
import os, sys, time, pickle, logging, datetime, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

# Point WW-DHSVM at a shared data cache.  Every manager writes here, and
# re-running this notebook re-downloads nothing.
os.environ.setdefault('WW_DHSVM_DATA_DIR', os.path.expanduser('~/ww_dhsvm/data'))
sys.path.insert(0, os.path.expanduser('~/ww_dhsvm'))

import ww_dhsvm
import ww_dhsvm.sources as S
import ww_dhsvm.grid as G
import ww_dhsvm.crs as C
import ww_dhsvm.terrain as T
import ww_dhsvm.streams as ST
import ww_dhsvm.soils as SO
import ww_dhsvm.vegetation as V
import ww_dhsvm.binary as B
import ww_dhsvm.states as STA
import ww_dhsvm.meteorology as M
import ww_dhsvm.config_writer as CW
import ww_dhsvm.diagnostics as D
import ww_dhsvm.output as O
import ww_dhsvm.workflow as W
import ww_dhsvm.figures as FIG
import ww_dhsvm.plot as PL

# Verbose logging is the point: every stage reports what it did, what it
# changed, and what it assumed.  The notebook logs through the same handler
# the library does, so its own commentary and the modules' internals arrive
# interleaved and in order -- which is what you want when a stage takes a
# minute and you are wondering whether it is still alive.
logging.basicConfig(level=logging.INFO, format='%(asctime)s  %(message)s',
                    datefmt='%H:%M:%S', force=True)
log = logging.getLogger('notebook')

PL.useStyle()
plt.rcParams['figure.figsize'] = (12, 5)

log.info(f'WW-DHSVM {ww_dhsvm.__version__}')

## Parameters

Everything a user would normally change lives in this one cell. Point it
at a different HUC — or a shapefile — and the rest of the notebook runs
unchanged.

A word on the two most consequential choices:

**Grid resolution.** DHSVM is normally run at 30–150 m. Finer resolves
hillslope convergence better but costs quadratically, and below ~10 m the
three-layer soil column and the saturated-lateral-flow parameterisation
stop being physically defensible. At 29,186 km², **150 m** gives
1,297,159 active cells on a 3019 × 1071 array — a large but tractable
DHSVM domain, and one that still runs a full year in under half an hour.

**Channel-initiation threshold.** The support area above which a cell is
treated as channel rather than hillslope. This is a genuine modelling
choice, not a technicality: it decides how much of the basin routes as
overland/subsurface flow versus channel flow. We check the result against
the mapped NHD drainage density below.

In [ ]:
# --- parameters ----------------------------------------------------------
NAME      = 'ConnecticutRiverBasin'
HUC       = '0108'            # USGS HUC4 -- the whole Connecticut River Basin, 29,186 km^2
# SHAPEFILE = '/path/to/basin.shp'   # the alternative entry point

CELLSIZE   = 150.0            # model grid spacing, m
TIMESTEP_H = 3.0              # DHSVM timestep, hours
START      = datetime.datetime(2025, 1, 1, 0)
END        = datetime.datetime(2025, 12, 31, 0)

MET_SOURCE            = 'AORC'   # 'AORC' | 'HRRR' | 'DayMet'
CHANNEL_THRESHOLD_KM2 = 0.5      # channel initiation support area
MIN_CHANNEL_SLOPE     = 1e-4     # m/m; see the note on filled valley floors
MAX_STATIONS          = 64       # forcing stations DHSVM interpolates from
SOIL_MIN_DEPTH        = 0.8      # m
SOIL_MAX_DEPTH        = 3.0      # m

EXPECTED_ANNUAL_PRECIP_MM = 1200.0   # New England climatology, for QA

CACHE  = os.path.expanduser('~/ww_dhsvm/demo/cache_ct')
ROOT   = os.path.expanduser(f'~/ww_dhsvm/demo/{NAME}')
dirs   = W.makeCaseDirectories(ROOT)
log.info('\n'.join(f'{k:>7}: {v}' for k, v in dirs.items()))

## Data sources

A *source manager* wraps one public dataset. It knows that dataset's
native CRS and resolution, buffers and snaps the request bounding box,
caches the result, and — importantly — detects when an earlier, larger
download already contains what is being asked for now.

This is the piece inherited wholesale from Watershed Workflow, and it is
why the second run of this notebook takes seconds rather than an hour.

In [ ]:
sources = S.getDefaultSources()
S.logSources(sources)

log.info('\nMeteorology options:')
for k, v in S.met_sources.items():
    sub = 'sub-daily' if S.met_is_subdaily[k] else 'DAILY ONLY'
    log.info(f'   {k:<8s} {v.name:<24s} {sub:<12s} {v.source}')

---
# Stage 1 — The domain

Two things happen here, and the second is the one that matters most for
the rest of the setup.

**Resolve the watershed.** `getWatershed` queries the USGS Watershed
Boundary Dataset for the hydrologic unit named by the HUC code. Pass a
string, never an integer — leading zeros are significant, and `010802`
is not `10802`.

**Lay out the model grid.** DHSVM runs on one regular, north-up,
square-celled grid in a projected CRS, and every subsequent input is
defined on it. `ModelGrid.fromShape` picks the UTM zone containing the
basin centroid, snaps the origin to a whole multiple of the cell size
(so the grid is reproducible and two runs at the same resolution share
cell edges), and pads by a few cells so the basin boundary never sits on
the array edge where DHSVM's routing stencil would be truncated.

### The grid convention, stated once

DHSVM's `[AREA]` section names the **north-west corner**, as
`Extreme West` and `Extreme North`. Cell centres sit at

```
x = Xorig + (col + 0.5) · DX
y = Yorig − (row + 0.5) · DY
```

so **row 0 is the northernmost row** — the same order the binary maps are
written in, and the same order a standard GDAL north-up transform uses.
This was verified against `InitConstants.c` and `InitMetSources.c` in the
DHSVM source, and against the corner coordinates of DHSVM's own Chiwawa
test case.

In [ ]:
# Stage 1 -- watershed boundary and model grid
watershed = ww_dhsvm.getWatershed(HUC, sources)
# ...or, for a user-supplied polygon:
# watershed = ww_dhsvm.getWatershedFromShapefile(SHAPEFILE)

display(watershed[['ID', 'name', 'area']] if 'area' in watershed
        else watershed.drop(columns=watershed.geometry.name).head())

grid = G.ModelGrid.fromShape(watershed, cellsize=CELLSIZE, buffer_cells=3)
grid.setMaskFromShape(watershed)
log.info('')
log.info(grid.summary())

In [ ]:
# Reference hydrography: NHD flowlines.  These are NOT used to define
# DHSVM's channel topology -- that comes from the conditioned DEM -- but
# they are used to burn the DEM and to validate the derived network.
nhd_path = os.path.join(CACHE, 'nhd.gpkg')
if os.path.exists(nhd_path):
    reaches = gpd.read_file(nhd_path)
else:
    reaches = sources['hydrography'].getShapesByGeometry(
        watershed.union_all(), watershed.crs, out_crs=grid.crs)
    # GeoPackage holds one geometry column; NHD returns flowline + catchment.
    extra = [c for c in reaches.columns
             if reaches[c].dtype.name == 'geometry' and c != reaches.geometry.name]
    reaches = reaches.drop(columns=extra) if extra else reaches
log.info(f'{len(reaches):,} NHD reaches, total length '
         f'{reaches.to_crs(grid.crs).length.sum()/1000:,.0f} km')

---
# Stage 2 — Terrain

The conditioned DEM is the single most influential input in a DHSVM
setup. DHSVM routes both surface and saturated subsurface water down the
gradient between neighbouring cells, so:

* a **sink** left in the DEM becomes a permanent lake that swallows
  everything routed into it;
* a **flat** becomes a place where the gradient is zero and water simply
  stops moving.

Three steps, in order:

**Resample.** 3DEP at 30 m is averaged onto the 150 m model grid. Using a
finer source than the model grid is deliberate — bilinear resampling of
30 m data gives a better 150 m mean elevation than fetching 60 m data.

**Burn.** Elevations along mapped NHD flowlines are lowered by 5 m before
filling. This forces the derived channel network to follow rivers that
actually exist. It matters most in low-relief valleys — like the
Connecticut main stem — where a 30 m DEM does not resolve the channel and
an unburned delineation wanders off across the floodplain.

**Fill and route.** Depressions are filled to their spill elevation
(Wang & Liu, 2006), then D8 flow directions and flow accumulation are
derived, along with slope and aspect by **Horn's** third-order finite
difference — the method ArcGIS and GRASS use, and the one the legacy
DHSVM AML preprocessing assumed.

**Confine routing to the basin.** The DEM is fetched with a buffer around
the watershed polygon, because slope and aspect on the cells just *inside*
the divide need the cells just outside it. Accumulate flow over that whole
raster, though, and every out-of-basin cell that happens to drain towards
the basin adds its area to yours. On the Connecticut that inflated the
outlet from 29,186 to **35,127 km²** — a fifth of the main stem's drainage
arriving from outside its own watershed, and with it a channel too wide
and too deep all the way down. `conditionDEM(..., confine_to_basin=True)`
masks the conditioned DEM to the basin before routing and warns if the
largest contributing area still exceeds basin area by more than 2%.

> **Note on D8 vs D4.** These D8 directions are used to *delineate the
> channel network*. They are not DHSVM's runtime routing scheme — DHSVM
> defaults to D4 for surface/subsurface routing and can be compiled for
> D8. D8 is the right choice for channel delineation either way.

In [ ]:
# Stage 2 -- DEM acquisition and conditioning
t0 = time.time()

def cached(name, fetch):
    # Load a gridded input from the demo cache, or fetch it live.
    #
    # The cache exists only so this notebook re-runs in seconds.  Delete
    # it, or point HUC at a different basin, and every input is fetched
    # live through the source managers -- which is the normal path.
    path = os.path.join(CACHE, name + '.npy')
    if os.path.exists(path):
        log.info(f'  {name}: from cache')
        return np.load(path).astype('float64')
    log.info(f'  {name}: fetching live ...')
    arr = fetch()
    os.makedirs(CACHE, exist_ok=True)
    np.save(path, arr.astype('float32'))
    return arr.astype('float64')

def _fetch_dem():
    ds = sources['DEM'].getDataset(grid.polygon().buffer(4 * CELLSIZE), grid.crs)
    return T.resampleToGrid(ds[list(ds.data_vars)[0]], grid, 'bilinear')

dem_raw = cached('dem_raw', _fetch_dem)
log.info(f'raw 3DEP DEM: {np.nanmin(dem_raw):.1f} to {np.nanmax(dem_raw):.1f} m, '
         f'{int(np.isnan(dem_raw).sum()):,} nodata cells')

# DHSVM binary maps have no nodata convention, so every in-basin cell must
# carry a real value.
dem_raw = T.fillGaps(dem_raw, grid.mask)
dem_raw = np.nan_to_num(dem_raw, nan=float(np.nanmedian(dem_raw)))

terrain = T.conditionDEM(dem_raw.astype('float32'), grid,
                         streams=reaches, burn_depth=5.0)
dem = terrain['dem']
log.info(f'\nconditioning took {time.time()-t0:.0f}s')

In [ ]:
# Diagnostic statistics for every terrain field
for name, arr, unit in [('elevation', terrain['dem'], 'm'),
                        ('slope', np.degrees(terrain['slope']), 'deg'),
                        ('upstream area', terrain['uparea'] / 1e6, 'km^2')]:
    s = T.maskedStats(arr, grid.mask, name)
    log.info(f"{name:>14s} [{unit:>5s}]  min {s['min']:9.2f}  p05 {s['p05']:9.2f}  "
             f"median {s['p50']:9.2f}  p95 {s['p95']:9.2f}  max {s['max']:9.2f}")

In [ ]:
fig = FIG.domainOverview(grid, watershed, dem, reaches); plt.show()

In [ ]:
fig = FIG.terrainPanel(grid, terrain); plt.show()

In [ ]:
fig = FIG.flowRoutingPanel(grid, terrain); plt.show()

---
# Stage 3 — The channel network

DHSVM represents channels as a **1D network of segments superimposed on
the 2D grid**, described by three files that must agree exactly:

**`stream.class.dat`** — one row per channel *class*:
`ID  width  depth  manning_n  infiltration`

**`stream.network.dat`** — one row per *segment*, defining topology:
`ID  order  slope  length  class  outlet_ID  [SAVE "name"]`
`outlet_ID` is the downstream segment, or `0` at the basin outlet.
Slope and length must both be **strictly positive** — `channel.c` treats
zero or negative as fatal.

**`stream.map.dat`** — one row per (cell, segment) intersection:
`col  row  seg_ID  length  cut_height  cut_width  aspect`
with **zero-based** `col` and `row`, row 0 northernmost.

### Where the network comes from

Watershed Workflow builds a *vector* river network for stream-aligned
meshing. DHSVM instead needs a network that is **consistent with the
routing grid** — a channel cell must be a cell water actually flows
through. So WW-DHSVM derives the network from the conditioned DEM's D8
flow directions, splits it into segments at confluences, and computes
each segment's slope, length and contributing area from the grid itself.
NHD is used to burn the DEM and to *check* the answer, never to define it.

### Channel geometry

Width and depth come from downstream hydraulic geometry — power laws in
contributing area, `W = 2.4·A_km²^0.45` and `D = 0.27·A_km²^0.30`, the
humid-temperate form. These give 2.4 m × 0.27 m at 1 km², 19 m × 1.1 m at
100 km², and 229 m × 5.6 m for the whole Connecticut at 25,000 km² —
which is about right for the river at Thompsonville. They are *regional*
relations and deserve recalibration wherever bankfull surveys exist.

In [ ]:
# Stage 3 -- delineate the channel network
network = ST.extractNetwork(terrain, grid,
                            channel_threshold_km2=CHANNEL_THRESHOLD_KM2,
                            min_slope=MIN_CHANNEL_SLOPE)

# Validate the topology BEFORE DHSVM sees it: duplicate IDs, dangling
# outlets, routing cycles, non-positive slope or length, reachability.
topology = ST.checkTopology(network['segments'])
log.info(f"\ntopology ok: {topology['ok']}")

### One NaN in one station ruins the whole basin

Worth stating before the forcing section, because it cost 17% of this
simulation before it was caught. AORC v1.1 has isolated missing hours:
over this basin in 2025, **one of 33 grid cells had NaN precipitation at
exactly two timesteps**, during a late-October storm.

DHSVM interpolates its forcing from *every* station at *every* timestep,
so those two NaNs propagated across the entire basin. Everything —
precipitation, soil moisture, ET, water table, discharge — went NaN from
`10/31 18:00` onward and stayed NaN for the rest of the year. DHSVM
printed no warning and **exited 0**.

`meteorology.writeStationFile` therefore repairs short gaps by time
interpolation before writing, logs every fill, and *refuses* gaps longer
than eight timesteps rather than inventing that much data. The
`summarize` table also counts missing values per variable, so the problem
is visible before the model ever runs.

### The `order` column is a routing rank, not Strahler order

This is the single subtlest thing about `stream.network.dat`, and getting
it wrong loses most of the water in the channel network without DHSVM
reporting an error.

`channel_route_network` in `channel.c` sweeps the segment list by
ascending `order`:

```c
for (order = 1; ; order += 1) {
    for every segment with current->order == order:
        channel_route_segment(...);   // adds its outflow into
                                      // segment->outlet->inflow
    if (no segment had this order) break;
}
```

A segment is routed at its own rank, and only *then* is its outflow added
to its downstream neighbour's inflow. For that to conserve mass, every
segment must be routed **strictly after** all of its upstream
contributors — so the column has to satisfy

$$\mathrm{order}(s) = 1 + \max_{u \,\in\, \mathrm{upstream}(s)} \mathrm{order}(u)$$

with headwaters at 1. **Strahler order does not satisfy this.** Two
order-2 reaches merging give order 3, but an order-1 reach joining an
order-3 leaves it order-3 — so a contributor and its receiver share a
rank, the receiver is routed in the same sweep (often *before* its
contributor), and because `channel_step_initialize_network` zeroes
`inflow` every timestep, the late contribution is silently dropped.

The symptom, on an early build of this workflow: **86 mm/yr of lateral
inflow entered the channels and 1.6 mm/yr came out**, with the rest
absorbed by DHSVM's channel closure error. Cell-level mass balance was perfect the
whole time (2×10⁻⁴ mm), because the loss is entirely inside channel
routing.

DHSVM's own Chiwawa test case settles the question: its `order` column is
`1 + max(upstream)` for all 494 segments and runs up to **69** — far
beyond any Strahler value. On the Connecticut the ranks reach **673**
across 23,790 segments, where Strahler order stops at 7.

WW-DHSVM therefore computes both, and keeps them separate: `order` in
the file is the routing rank, while `strahler_order` — the hydrologically
meaningful one — is carried alongside for classification and plotting.

### Channel slope on a filled valley floor — a trap worth knowing

Depression filling raises every cell in a depression to its spill
elevation, so a reach lying inside one has **exactly zero drop** — not a
small drop, zero. Along a low-relief valley floor like the Connecticut's
that is a quarter of the network here.

That matters because DHSVM routes channel water with Manning's equation,
in which velocity scales with the square root of slope. Left at a token
value like 1e-5 (1 cm per km) those reaches transport essentially
nothing: water routed into them accumulates rather than reaching the
outlet, and DHSVM absorbs the difference in its channel-network closure
error. In an early build, 104 mm/yr of lateral inflow entered the
channels and **1.8 mm/yr came out**.

So `extractNetwork` does not simply clamp. For each reach below
`min_slope` it walks *downstream*, accumulating length and drop until the
accumulated gradient reaches the threshold, and assigns that — the local
**valley** slope, which is the physically meaningful quantity. A reach on
a filled flat really does sit in a valley that descends; the DEM just
fails to resolve it over one reach length. Only reaches for which even
the whole downstream path stays flatter take the floor.

For this basin: 5,896 reaches (24.8%) came out below 1e-4, of which
**5,723 recovered a real gradient** from the valley and only 173 (0.7%)
were floored.

### Two more channel traps, both specific to large basins

**Channel classes must scale with the basin.** PNNL's `channelclass.py`
bins reaches by contributing area and its top band is simply
`row[1] > 40000000` — everything above 40 km², one class. That is right
for the ~500 km² Chiwawa it was written for. Applied unchanged to a
29,186 km² basin it puts **everything from 40 km² to the 29,145 km²
outlet into a single class**, one width and one depth across three orders
of magnitude of drainage area. DHSVM multiplies channel length by width
to get interception area, so the error lands in the water balance.

`classifyChannels` therefore derives its bands from *this* basin's area
distribution, **log-spaced**. Quantile spacing was tried first and is
wrong: contributing area is power-law distributed, so quantiles pile five
of six breaks below 35 km² and leave one band spanning 35 km² to the
outlet. Log spacing bounds the width error across the whole network.

**…and then DHSVM never clamps channel area to cell area.** Fix the
widths and you meet the next problem. `ChannelCut` sets a cell's channel
area to `width × length` with no bound. Hydraulic geometry gives this
outlet 245 m of width on a **150 m** grid, and even the class
representative is 162.6 m — 108% of a cell. An earlier build of this
basin duly produced an impossible water balance: channel interception of
**1,153 mm against 955 mm of precipitation**, and a runoff ratio of
**1.19**.

The real constraint is a resolution limit — *a grid cannot represent a
river wider than its cells*. `_capChannelAreaToCell` scales widths down
where a cell is over-subscribed, keeps at least 10% of the cell as land
to generate runoff, and warns that a finer grid is the actual fix. On
this basin it binds on 2,598 of 138,645 channel cells (1.9%) — the main
stem only.

### Choosing the threshold

The threshold is a real modelling decision, so it is worth seeing its
effect. Sweeping it over this basin against the NHD medium-resolution
network (0.834 km/km²):

On this basin, a 0.5 km² threshold gives 23,790 segments,
138,645 channel cells and a drainage density of **0.713 km/km²** against
NHD's **0.834** — 86% of the mapped density, with 73% of derived channel
cells falling within one cell of a mapped reach.

Lowering the threshold raises the density toward NHD's, but NHD medium
resolution includes ephemeral streams that DHSVM would then route as
perennial channels. 0.5 km² is a deliberate middle: dense enough to be
realistic, without asserting that every mapped ephemeral line carries
water year-round.

In [ ]:
# Is the derived network as dense as the mapped one?  A large mismatch
# means the channel-initiation threshold needs revisiting.
comparison = ST.compareToReference(network['channel_mask'], reaches, grid)

display(network['segments'].head(8))
log.info('\nChannel classes actually used:')
display(network['class_table'])

In [ ]:
fig = FIG.channelPanel(grid, network, terrain, comparison); plt.show()

---
# Stage 4 — Soils

DHSVM discretises the subsurface as a small number of **soil types**,
each with ~20 parameters, plus two maps: a soil-type map and a total
soil-depth map.

The parameterisation is **Brooks–Corey**, not van Genuchten — this is one
of the concrete places the ATS fork had to be rewritten rather than
ported:

* `Porosity` — saturated water content φ
* `Pore Size Distribution` — Brooks–Corey λ (= 1/b in Clapp & Hornberger)
* `Bubbling Pressure` — air-entry head ψ_b, **metres**
* `Field Capacity` / `Wilting Point` — θ at −33 and −1500 kPa
* `Vertical Conductivity` / `Lateral Conductivity` — K_sat, m/s
* `Exponential Decrease` — *f*, the decay of lateral K with depth

### The two parameters that matter most

`Lateral Conductivity` and `Exponential Decrease` together control
saturated subsurface flow — the mechanism that generates most baseflow
and much of the storm response. They set recession behaviour, and they
are routinely **calibrated**. The values assigned here from texture are a
defensible starting point, not an answer.

### Soil depth

No national dataset gives DHSVM's total soil-column depth directly. The
community's standard substitute, which WW-DHSVM implements from PNNL's
`soildepth.aml`, is a weighted terrain index — soils are deep in flat,
low, high-accumulation positions and thin on steep, high, divergent ones:

$$d = d_{min} + (d_{max}-d_{min})\left[w_s\left(1-(S/S_{max})^{p_s}\right)
+ w_a (A/A_{max})^{p_a} + w_e\left(1-(E/E_{max})^{p_e}\right)\right]$$

This is a **terrain proxy, not a measurement**, and it is stated as such
in the code. It is then reconciled against two hard DHSVM constraints:
soil must be deeper than the channel cut (`channel_grid.c`), and deeper
than the local rooting depth (`CheckOut.c` treats a violation as fatal).

In [ ]:
# Stage 4 -- soil texture from POLARIS (30 m), classified on the USDA triangle
def _fetch_texture(sep):
    def go():
        # POLARIS reports six standard depth intervals; DHSVM wants one
        # texture class per cell, so collapse to a thickness-weighted mean
        # over the top metre -- the depth the model actually simulates.
        import xarray as xr
        ds = S.soil_sources['POLARIS'].getDataset(
            grid.polygon().buffer(600), grid.crs, variables=[sep])
        layered = ds[sep]
        flat = SO.aggregateDepthLayers(
            layered, max_depth_cm=100.0,
            depth_dim=[d for d in layered.dims if d not in ('x', 'y')][0]
            if layered.ndim == 3 else None)
        da = xr.DataArray(flat, dims=('y', 'x'),
                          coords={'y': layered['y'], 'x': layered['x']})
        return T.resampleToGrid(da.rio.write_crs(layered.rio.crs), grid, 'bilinear')
    return go

sand = cached('sand', _fetch_texture('sand'))
silt = cached('silt', _fetch_texture('silt'))
clay = cached('clay', _fetch_texture('clay'))
m = grid.mask != 0
log.info(f'basin-mean texture: sand {np.nanmean(sand[m]):.1f}%  '
         f'silt {np.nanmean(silt[m]):.1f}%  clay {np.nanmean(clay[m]):.1f}%')

soil_cls = SO.classifyFromFractions(sand, silt, clay, grid.mask)
soil_map, soil_table = SO.compactClasses(soil_cls, grid.mask)
soil_blocks = SO.buildSoilParameterBlocks(soil_table, n_layers=3)
SO.checkSoilConsistency(soil_blocks)

In [ ]:
# The DHSVM [SOILS] parameter table actually written to the config file
display(soil_table[['dhsvm_id', 'name', 'porosity', 'field_capacity',
                    'wilting_point', 'vertical_conductivity_ms',
                    'lateral_conductivity_ms', 'bubbling_pressure_m']])

In [ ]:
# Soil depth: terrain index, then reconciled with the channel cut depth
soil_depth = T.estimateSoilDepth(terrain['slope'], terrain['dem'],
                                 terrain['uparea'],
                                 min_depth=SOIL_MIN_DEPTH,
                                 max_depth=SOIL_MAX_DEPTH, mask=grid.mask)
soil_depth = T.enforceSoilDepthBelowChannels(soil_depth,
                                             network['cut_height_grid'])

---
# Stage 5 — Vegetation

DHSVM's canopy is a **two-layer** representation — an optional overstory
and an optional understory, each with its own height, LAI, albedo,
stomatal resistance and root distribution. Interception, snow
interception and unloading, radiation attenuation, aerodynamic resistance
and transpiration are all computed per layer. This is what makes DHSVM
well suited to forest-management questions, and it is why the vegetation
block is the largest section of a DHSVM config file — roughly **35 keys
per type**.

WW-DHSVM crosswalks **NLCD** to DHSVM vegetation types through an
editable table (`ww_dhsvm/data/nlcd_to_dhsvm_vegetation.json`). It *should* be
edited: canopy height, fractional coverage and minimum stomatal
resistance are regionally variable and among the most sensitive
parameters in the model.

**Seasonality** is carried by 12-element monthly arrays. The built-in
table uses phenologically realistic Northern-Hemisphere temperate
profiles — deciduous forest swings from LAI 0.2 in winter to 5.5 at
midsummer; evergreen holds 4.5–5.5 year-round. For the Connecticut
valley, that contrast drives most of the seasonal ET signal.

**Impervious area** from NLCD's percent-impervious product refines each
developed class's literature default to this basin's actual mean. Note
that as soon as *any* type has a non-zero impervious fraction, DHSVM
demands an impervious-surface routing file — generated below.

In [ ]:
# Stage 5 -- NLCD land cover, crosswalked to DHSVM vegetation types
def _fetch_nlcd(var, resampling, nodata=np.nan):
    def go():
        ds = sources['land cover'].getDataset(
            grid.polygon().buffer(4 * CELLSIZE), grid.crs, variables=[var])
        return T.resampleToGrid(ds[var], grid, resampling, nodata=nodata)
    return go

nlcd   = cached('nlcd', _fetch_nlcd('cover', 'nearest', nodata=0)).astype('uint8')
imperv = cached('impervious', _fetch_nlcd('impervious', 'bilinear'))

cover = V.classifyLandCover(nlcd, grid.mask)
veg_map, veg_table = V.compactClasses(cover, grid.mask)
veg_blocks = V.buildVegetationBlocks(veg_table)

# Refine each developed class's impervious fraction from the NLCD product
V.applyImperviousFraction(veg_blocks, imperv, veg_map, grid.mask)
V.checkVegetationConsistency(veg_blocks, n_soil_layers=3)

In [ ]:
# DHSVM (CheckOut.c) treats soil shallower than the root zone as FATAL.
# Reconcile now that vegetation is known.
soil_depth = T.enforceSoilDepthBelowRootZone(soil_depth, veg_map, veg_blocks,
                                             grid.mask)

awc = SO.availableWaterCapacity(soil_blocks, soil_map, soil_depth, grid.mask)

In [ ]:
fig = FIG.soilPanel(grid, soil_map, soil_table, soil_depth, awc); plt.show()

In [ ]:
fig = FIG.vegetationPanel(grid, veg_map, veg_table, veg_blocks); plt.show()

In [ ]:
# A look at the full DHSVM parameter block for the most extensive canopy type
top = veg_table.sort_values('n_cells', ascending=False).iloc[0]
b = next(x for x in veg_blocks if x['id'] == int(top['dhsvm_id']))
log.info(f"DHSVM vegetation type {b['id']}: {b['description']}  "
         f"(NLCD {b['nlcd_code']}, {top['n_cells']:,} cells)\n")
for k, v in b.items():
    if isinstance(v, list) and len(v) == 12:
        log.info(f'  {k:<32s} {" ".join(f"{x:.2f}" for x in v)}')
    elif isinstance(v, list):
        log.info(f'  {k:<32s} {v}')
    else:
        log.info(f'  {k:<32s} {v}')

---
# Writing the DHSVM input maps

Now the arrays become files. `binary.writeMap` selects each map's dtype
from the table transcribed out of DHSVM's `VarID.c` — get this wrong and
the file is the wrong length, and DHSVM silently reads past the end of
it. An ESRI ASCII twin is written alongside each binary so the inputs are
inspectable in QGIS.

Note the **impervious-surface routing file**: DHSVM requires it as soon
as any vegetation type is partly impervious. It lists, for every in-basin
cell in row-major order, the nearest channel cell that impervious runoff
is delivered to — which is what a storm-drain network does in reality.
DHSVM re-reads the first two fields of each line and aborts if they do
not match the cell it expects, so the ordering is not optional.

In [ ]:
case = dict(grid=grid, watershed=watershed, sources=sources, reaches=reaches,
            huc=HUC, terrain=terrain, dem=dem, network=network,
            soil_map=soil_map, soil_table=soil_table, soil_blocks=soil_blocks,
            soil_depth=soil_depth, veg_map=veg_map, veg_table=veg_table,
            veg_blocks=veg_blocks, canopy_gapping=False)

paths = W.writeMaps(case, dirs, also_ascii=True)
log.info('')
for k, v in paths.items():
    log.info(f'  {k:<20s} {os.path.getsize(v):>12,d} bytes   {v}')

In [ ]:
# Verify every binary map round-trips at the right shape and dtype
log.info('round-trip verification:')
for kind, key in [('dem', 'dem'), ('mask', 'mask'), ('soil', 'soil'),
                  ('soil_depth', 'soil_depth'), ('veg', 'veg')]:
    a = B.readMap(paths[key], grid.nrows, grid.ncols, kind)
    log.info(f'  {key:<12s} {str(a.shape):<14s} {a.dtype}  '
             f'range {a.min():.3f} .. {a.max():.3f}')

---
# Stage 6 — Meteorological forcing

DHSVM reads forcing as a set of **point time series** ("stations"), one
ASCII file each, and interpolates them onto the grid every timestep with
inverse-distance weights, applying lapse-rate corrections from each
station's elevation to each cell's elevation.

The station file format, verified against `ReadMetRecord.c`:

```
MM/DD/YYYY-HH   Tair   Wind   RH   Sin   Lin   Precip
```

| column | quantity | units |
|---|---|---|
| 2 | air temperature | **°C**, not Kelvin |
| 3 | wind speed | m/s |
| 4 | relative humidity | **percent** |
| 5 | incoming shortwave | W/m² |
| 6 | incoming longwave | W/m² |
| 7 | precipitation | **metres per timestep**, not a rate |

Two of those are easy to get wrong and worth repeating: temperature is
**Celsius**, and precipitation is a **depth accumulated over one model
timestep**. At a 3-hour step, 2.5 mm of rain is written `0.002500`.

### Why station mode, not gridded mode

DHSVM also offers `Gridded Met data = TRUE`, which scans a directory and
infers station positions from `prefix_LAT_LON` filenames. **WW-DHSVM
deliberately does not use it.** Reading `InitMetSources.c` shows the
gridded path never assigns `Stat[k].Elev`, so it stays at its `calloc`
value of zero — and `MakeLocalMetData.c` then lapses temperature from
*sea level* to every cell. For this basin, whose mean elevation is ~180 m,
the default −0.0065 °C/m lapse would impose a systematic **−1.2 °C cold
bias**, which would show up as far too much snow.

Station mode lets WW-DHSVM write an explicit `Elevation` for every
station, sampled from the same DEM the model runs on, so the lapse
correction is a small physically-meaningful adjustment instead of a large
spurious one.

### Time convention

Forcing is written in **UTC**, and the config sets `Time Zone Meridian =
0`. `CalcSolar.c` computes the offset from clock noon to solar noon as
`4 min/deg × (StandardMeridian − Longitude)`; with a zero meridian and
UTC timestamps this reduces to the site's true longitude offset, so solar
geometry is exact and daylight-saving never enters. Simpler *and* more
accurate than converting to local standard time.

### Which product

| | **AORC** | **HRRR** | **Daymet** |
|---|---|---|---|
| resolution | 1 km | **3 km** | 1 km |
| timestep | hourly | **hourly** | **daily only** |
| period | 1979– | 2014-07-30– | 1980– |
| nature | analysis of record | operational NWP analysis/forecast | interpolated observations |
| wind | yes | yes | **no — assumed** |
| longwave | yes | yes | **no — estimated** |

AORC drives this run: it is the calibrated analysis-of-record and the
right choice for an annual water balance. HRRR is fetched below for
comparison — it is the better choice for recent, event-scale,
high-resolution work, and it is the reason the HRRR manager exists.

In [ ]:
# Stage 6 -- fetch, convert and write the forcing
mgr = S.met_sources[MET_SOURCE]

# AORC is 1 km: a year of eight hourly variables over 29,186 km^2 is ~75 GB,
# while DHSVM -- which interpolates from a station list -- gains nothing past
# a few dozen stations.  Decimating to ~8 km keeps ACTUAL AORC cells (never
# averages), so every station stays physically self-consistent.
kw = dict(spatial_decimation=8, temporal_resampling='3h') if MET_SOURCE == 'AORC' else {}

met_raw = mgr.getDataset(grid.polygon(), grid.crs,
                         start=START.strftime('%Y-%m-%d'),
                         end=END.strftime('%Y-%m-%d'), **kw)
log.info(f'raw met dataset: {dict(met_raw.sizes)}')

met = M.convertToDHSVM(met_raw, MET_SOURCE)
if MET_SOURCE != 'AORC':
    met = met.resample(time=f'{int(TIMESTEP_H)}h').mean()

In [ ]:
# Physical plausibility of every driving variable
summary = M.summarize(met, TIMESTEP_H)
display(summary)

In [ ]:
# The number a hydrologist knows by heart: annual precipitation
met_summary = M.annualWaterBalanceCheck(met, TIMESTEP_H)

In [ ]:
# Place stations on the forcing dataset's OWN cell centres, so no spatial
# interpolation happens before DHSVM sees the data, and give each one its
# DEM elevation.
stations = M.placeStations(grid, dem, met, max_stations=MAX_STATIONS)
stations = M.writeAllStations(dirs['met'], met, stations, TIMESTEP_H)

log.info(f'\nfirst station file, first 3 rows:')
with open(stations[0].filename) as f:
    for _ in range(3):
        log.info('    ' + f.readline().rstrip())
log.info(f'\n  columns: date  Tair[C]  Wind[m/s]  RH[%]  Sin[W/m2]  Lin[W/m2]  '
         f'Precip[m per {TIMESTEP_H:g}h step]')

In [ ]:
fig = FIG.forcingPanel(met, stations, grid, dem, TIMESTEP_H); plt.show()

### Comparing AORC with HRRR

The HRRR manager is the one genuinely new data source in this fork. It
retrieves NOAA's 3 km convection-allowing model through
[Herbie](https://github.com/blaylockbk/Herbie), assembling each valid
hour from the run initialised one hour earlier at forecast hour 1 —
because accumulated precipitation does not exist in the analysis, and the
radiation fluxes are period averages. That gives a seamless hourly series
in which precipitation is a true 1-hour accumulation and the fluxes are
means over the preceding hour, which is exactly DHSVM's convention.

Comparing two independent products over the same basin is the cheapest
meaningful check on forcing quality. Over 1–14 June 2025 on this basin,
daily basin means give:

| variable | AORC | HRRR | HRRR − AORC | r |
|---|---|---|---|---|
| air temperature (°C) | 16.72 | 16.68 | **−0.04** | 0.979 |
| precipitation (mm/day) | 2.84 | 1.95 | **−0.90** | 0.951 |
| shortwave in (W/m²) | 242.5 | 266.2 | +23.7 | 0.864 |
| longwave in (W/m²) | 345.7 | 332.9 | −12.8 | 0.893 |
| relative humidity (%) | 75.9 | 71.2 | −4.8 | 0.953 |
| wind speed (m/s) | 2.20 | 3.08 | **+0.88** | 0.994 |

Correlations of 0.86–0.99 across two entirely independent products is
reassuring for both. The biases are the documented ones: HRRR runs
**windier**, and **drier on convective June rainfall** — its
precipitation is a short-range forecast field, so it places convection
imperfectly even when it gets the timing right (note the 7 June storm,
where the two agree on timing and differ by a fifth on depth). HRRR is
also sunnier and less humid, consistent with modelling a little too
little cloud.

For an annual water balance, that 31% precipitation difference is the
one that matters, and it is why this run is driven by AORC. For a storm
study at 3 km and hourly resolution, HRRR is the better instrument.

In [ ]:
# A representative month of HRRR, for comparison against AORC
try:
    hrrr_raw = S.met_sources['HRRR'].getDataset(
        grid.polygon(), grid.crs, start='2025-06-01', end='2025-06-30')
    hrrr = M.convertToDHSVM(hrrr_raw, 'HRRR').resample(
        time=f'{int(TIMESTEP_H)}h').mean()
    aorc_window = met.sel(time=slice('2025-06-01', '2025-06-30'))
    fig = FIG.forcingComparison(aorc_window, hrrr, 'AORC', 'HRRR', TIMESTEP_H)
    plt.show()
except Exception as exc:
    log.info(f'HRRR comparison skipped: {type(exc).__name__}: {exc}')

---
# Stages 7–8 — Initial state and configuration

### Initial state

DHSVM restores four files stamped with the model start time. The three
binary ones are **stacks of same-shaped float32 matrices**, and the
stacking order is fixed by the sequence of `Read2DMatrix` calls in
`InitModelState.c`. WW-DHSVM's layout was verified against the file sizes
distributed with DHSVM's own Chiwawa test case (425 × 300 cells,
510,000 bytes per matrix):

| file | layout | Chiwawa | matrices |
|---|---|---|---|
| `Interception.State` | `2·L_veg + 1` | 2,550,000 | 5 |
| `Snow.State` | 8, fixed | 4,080,000 | 8 |
| `Soil.State` | `2·L_soil + 4` | 5,100,000 | 10 |

`Channel.State` is plain text — and note it carries **no** `.bin`
extension, unlike the other three.

> **These are a cold start, not an equilibrated state.** Soil moisture is
> set to 90% of field capacity, there is no snow, and channels hold no
> storage. Subsurface storage in a DHSVM basin typically needs **1–3
> years** to stop drifting. The right pattern for real work is to run a
> spin-up, let DHSVM dump its own state via `Number of Model States`, and
> restart the analysis run from that. The water balance below shows this
> run still filling storage.

### Configuration

The config writer handles the cross-dependencies between options and
required keys — `Flow Routing = NETWORK` needs the three stream files;
station mode needs a five-key block per station; `Number of Soil Types`
must match the `[SOILS]` blocks *and* bound every value in `soil.bin`;
`Number of Root Zones` must equal `Number of Soil Layers`;
`Reference Height` must exceed the tallest canopy. `validateConfig`
checks all of these against the actual maps before DHSVM ever runs.

In [ ]:
# Stage 7 -- initial model state
case.update(paths=paths, met=met, stations=stations,
            timestep_hours=TIMESTEP_H, met_summary=met_summary)

state_files = STA.writeInitialStates(
    dirs['state'], START, grid, soil_map, soil_blocks,
    network['segments']['ID'].tolist(),
    n_veg_layers=V.maxVegLayers(veg_blocks),
    n_soil_layers=soil_blocks[0]['n_layers'])

# A wrong matrix count is the commonest cause of a restart reading garbage:
# DHSVM does not check, it just seeks to an offset and reads.
STA.verifyStateFiles(state_files, grid,
                     n_veg_layers=V.maxVegLayers(veg_blocks),
                     n_soil_layers=soil_blocks[0]['n_layers'])
case['state_files'] = state_files

In [ ]:
# Stage 8 -- assemble and validate the DHSVM configuration file
cfg = CW.buildConfig(NAME, grid, START, END, TIMESTEP_H, paths,
                     soil_blocks, veg_blocks, stations,
                     output_dir=dirs['output'], state_dir=dirs['state'],
                     n_soil_layers=soil_blocks[0]['n_layers'])

validation = CW.validateConfig(cfg, grid, soil_blocks, veg_blocks,
                               soil_map, veg_map, TIMESTEP_H)
config_file = cfg.write(os.path.join(dirs['root'], f'INPUT.{NAME}'))
case.update(config=cfg, config_file=config_file, config_validation=validation)

In [ ]:
# The generated configuration, first 90 lines
with open(config_file) as f:
    log.info(''.join(f.readlines()[:90]))

---
# Stage 9 — Input diagnostics

DHSVM's own input validation is thin, and its error messages are numeric
codes emitted after several minutes of initialisation. This report runs
the checks *first*, in three tiers:

**Structural** — things DHSVM will reject outright: a soil class above
`Number of Soil Types`, a segment whose outlet does not exist, a state
file with the wrong matrix count, soil shallower than the root zone.
These are hard errors.

**Physical** — things DHSVM will happily run with but which are almost
certainly wrong: a wilting point above field capacity, annual
precipitation half what the region receives, a drainage density an order
of magnitude off the mapped network. Warnings that deserve a decision.

**Descriptive** — the numbers you want anyway. Not pass/fail, but the
fastest way to notice the domain is not the one you meant to build.

In [ ]:
report = D.checkInputs(grid, dem, soil_map, soil_depth, veg_map,
                       soil_blocks, veg_blocks,
                       network=network, met_summary=met_summary,
                       expected_annual_precip_mm=EXPECTED_ANNUAL_PRECIP_MM)
case['report'] = report
log.info(report.render())

In [ ]:
fig = FIG.inputDashboard(case); plt.show()

---
# Running DHSVM

Two builds live in `~/ww_dhsvm/build`:

* **serial** — DHSVM 3.2 from the PNNL `master` branch;
* **parallel** — the `parallel` branch, which distributes the grid across
  MPI ranks using [Global Arrays](https://hpc.pnl.gov/globalarrays/)
  ([Perkins et al., 2019](https://doi.org/10.1016/j.envsoft.2019.104533)).

This basin has 1,297,159 active cells and a year at a 3-hour timestep is
2,913 steps, so the run goes to the **parallel build on 8 ranks** — about
23 minutes.

> **Why the whole basin, and not a piece of it.** An earlier version of
> this notebook modelled HUC6 `010802` ("Lower Connecticut") alone. That
> is a *hydrologic unit*, not a closed watershed: the main stem enters
> across its northern boundary carrying the discharge of everything
> upstream, and DHSVM applies **no external inflow** at a domain
> boundary. The simulated hydrograph was therefore missing the upper
> basin entirely.
>
> HUC4 `0108` is the whole basin — every drop that leaves it at Old
> Saybrook fell inside it — so the outlet hydrograph is a complete
> water balance and is directly comparable to a gauge.

In [ ]:
import subprocess, shutil

DHSVM_PARALLEL = os.path.expanduser('~/ww_dhsvm/build/dhsvm-parallel/DHSVM/sourcecode/DHSVM')
DHSVM_SERIAL   = os.path.expanduser('~/ww_dhsvm/build/dhsvm-serial/DHSVM/sourcecode/DHSVM')
NPROC = 8

exe = DHSVM_PARALLEL if os.path.exists(DHSVM_PARALLEL) else DHSVM_SERIAL
cmd = (['mpirun', '-np', str(NPROC), exe, config_file]
       if exe == DHSVM_PARALLEL else [exe, config_file])
log.info('running: ' + ' '.join(cmd))

t0 = time.time()
proc = subprocess.run(cmd, cwd=dirs['root'], capture_output=True, text=True)
elapsed = time.time() - t0
log.info(f'\nexit code {proc.returncode}   wall time {elapsed/60:.1f} min')
log.info('\n'.join(proc.stdout.splitlines()[-12:]))

In [ ]:
# DHSVM writes its closing water balance to stderr
log.info('\n'.join(proc.stderr.splitlines()[-24:]))

---
# Results

DHSVM writes ASCII time series into its output directory. Three different
timestamp spellings appear across the files, and channel discharge is in
**cubic metres per model timestep** rather than m³/s — `ww_dhsvm.output`
normalises all of it and converts discharge given the timestep.

| file | content |
|---|---|
| `Aggregated.Values` | ~60 basin-average state and flux variables |
| `Mass.Balance` | per-timestep water-balance terms |
| `Streamflow.Only` | outflow of every `SAVE` segment |
| `Stream.Flow` | per-segment inflow / lateral / outflow / storage, **plus** a network-totals row per timestep with a *different* seven-field layout |
| `saturation_extent.txt` | saturated fraction of the basin |

Two residuals are worth looking at:

The **cell mass-balance residual** (`Mass.Balance`, column `Error`) should
sit at machine-noise level; a growing one means a real bug or an unstable
timestep.

The **channel-network closure error** (the totals row of `Stream.Flow`)
should be small against the lateral inflow. A large one means water is
disappearing inside the channel routing — almost always because reaches
have too little gradient to move it. `waterBalanceSummary` reports both,
and takes basin outflow from that totals row rather than by summing the
`SAVE`-flagged segments, since only the totals row accumulates over
*every* outlet.

In [ ]:
results = O.readAll(dirs['output'], timestep_hours=TIMESTEP_H)
log.info(f'\nfiles read: {list(results)}')

In [ ]:
water_balance = O.waterBalanceSummary(results, grid.basin_area_km2, TIMESTEP_H)

In [ ]:
out_report = D.compareOutputToExpectation(water_balance, grid.basin_area_km2)
log.info(out_report.render())

Two panels below repay a second look.

The **flow duration curve** is the one summary that judges the whole flow
regime rather than the peaks. A model with no baseflow shows it here as a
cliff at the dry end long before the hydrograph looks wrong.

The **water balance** is drawn as a waterfall of `MassBalance.c`'s own
terms — `Precip + vapour fluxes = ET + ChannelInt + RoadInt + Δstorage` —
and not a balance of our own devising. That distinction is not pedantry:
an earlier version of this notebook constructed its own balance, put
routed outlet discharge where `ChannelInt` belongs, and reported 149 mm
of "unaccounted" water that did not exist. What leaves the hillslopes is
`ChannelInt`; what leaves the *network* is discharge; the two differ by
whatever the channels are still holding.

In [ ]:
fig = FIG.outputPanel(results, grid, TIMESTEP_H, water_balance); plt.show()

In [ ]:
# Monthly water balance -- the seasonal signature of a snow-influenced
# temperate basin: winter accumulation, a spring melt peak, a summer ET
# maximum that draws storage down.
agg = results['aggregated']
mb  = results['mass_balance']
monthly = pd.DataFrame({
    'precipitation': mb['Precip(m)'].resample('MS').sum() * 1000,
    'evapotranspiration': mb['TotalET'].resample('MS').sum() * 1000,
    'snow water equivalent': agg['Swq'].resample('MS').mean() * 1000,
})
display(monthly.round(1))

---
# How well does it parallelize?

`demo/scaling.py` sweeps 1 → 14 MPI ranks (90% of this machine's 16
physical cores) in two experiments on this basin.

**Strong scaling** fixes the 150 m problem and adds ranks. **Weak
scaling** holds work per rank constant by setting
`cellsize(p) = 600 m / √p`, building an independent case at each
resolution — active cells per rank come out constant to four significant
figures (81,060 to 81,073).

Two measurement choices change the answer: DHSVM's own `Runtime Summary`
uses `clock()`, which is CPU time and counts MPI busy-waiting, so wall
time is measured externally; and each configuration is run twice, at 8
and 240 timesteps, so initialization can be differenced out from the
stepping loop that actually parallelizes.

| ranks | strong s/step | speedup | weak s/step | weak eff. |
|---|---|---|---|---|
| 1 | 1.249 | 1.00 | 0.083 | 100% |
| 2 | 0.820 | 1.52 | 0.103 | 80% |
| 4 | 0.597 | 2.09 | 0.144 | 58% |
| **6** | **0.508** | **2.46** | 0.192 | 43% |
| 8 | 0.531 | 2.35 | 0.263 | 32% |
| 14 | 0.605 | 2.06 | 0.538 | 15% |

Speedup peaks at **6 ranks (2.5×)**. The tempting explanation is DHSVM's
channel routing, whose serial sweep depth is the routing rank and grows
from 177 to ~900 as the weak grid refines — but the data rejects it:
step time correlates with routing rank at only *r* = 0.67 against
*r* = 0.99 for the rank count itself, and in a joint fit the routing-rank
term takes the wrong sign.

What fits is the simplest model available — fixed work split `p` ways
plus an overhead paid once per rank per timestep, `T(p) = W/p + c·p`:

| | W (s) | c (ms/rank) | R² |
|---|---|---|---|
| strong | 1.285 | **42.1** | 0.918 |
| weak | 0.044 | **35.7** | 0.983 |

Two independent experiments recover the same overhead coefficient to
within 15%, and the model predicts the strong-scaling optimum at
`√(W/c) = 5.5` where the sweep measured **6**.

> **Caveat.** This is one laptop-class node under WSL2 with all ranks on
> one memory controller — not a cluster.
> [Perkins et al. (2019)](https://doi.org/10.1016/j.envsoft.2019.104533)
> report considerably better scaling on HPC hardware. The portable lesson
> is the method, not the number: measure `W` and `c` on your machine and
> run at `√(W/c)`.

In [ ]:
# Reproduce the sweep (takes ~40 min: 16 timed runs plus 8 case builds)
# !python demo/scaling.py && python demo/plot_scaling.py
from IPython.display import Image
Image(os.path.expanduser('~/ww_dhsvm/demo/figures/12_scaling.png'))

---
# Where to go from here

**This case is a defensible first simulation, not a calibrated model.**
Everything it asserts is visible in the diagnostics above, and the
assumptions worth revisiting, in rough order of leverage:

1. **Spin-up.** The water balance above shows storage still filling from
   the cold start. Run 2–3 years, dump a state, restart from it.
2. **Lateral conductivity and exponential decrease.** These set baseflow
   recession and are the standard first calibration targets. They are
   currently assigned from texture alone.
3. **Soil depth.** A terrain proxy, not a measurement. Where a credible
   depth-to-bedrock product exists, blend it in with
   `terrain.blendSoilDepth`.
4. **Channel geometry.** Regional hydraulic-geometry relations. Recalibrate
   against bankfull surveys or HYDRoSWOT where available.
5. **Vegetation parameters.** Canopy height, fractional coverage and
   minimum stomatal resistance are regionally variable; edit
   `ww_dhsvm/data/nlcd_to_dhsvm_vegetation.json`.
6. **Channel-initiation threshold.** Compare the derived drainage density
   against NHD (done above) and adjust.

**To build a different basin**, change `HUC` in the parameters cell —
or set `SHAPEFILE` and swap the one call in Stage 1. Everything
downstream, including the source managers' caching, follows.